# खर्च दावी विश्लेषण

यो नोटबुकले कसरी एजेन्टहरू सिर्जना गर्ने देखाउँछ जसले प्लगइनहरू प्रयोग गरी स्थानीय रसिद छविहरूबाट यात्रा खर्चहरू प्रशोधन गर्छन्, खर्च दावी इमेल उत्पन्न गर्छन्, र खर्च डाटा पाई चार्ट प्रयोग गरी दृश्य प्रदान गर्छ। एजेन्टहरूले कार्य सन्दर्भको आधारमा गतिशील रूपमा कार्यहरू छनौट गर्छन्।

पटकथाहरू:
1. OCR एजेन्टले स्थानीय रसिद छविलाई प्रशोधन गरी यात्रा खर्च डाटा निकाल्छ।
2. इमेल एजेन्टले खर्च दावी इमेल उत्पन्न गर्छ।

### यात्राको खर्च परिदृश्यको उदाहरण:
कल्पना गर्नुहोस् तपाईँ व्यवसायिक बैठकका लागि अर्को शहरमा यात्रा गर्दै हुनुहुन्छ। तपाईँको कम्पनीले सबै उचित यात्रा सम्बन्धी खर्चहरूको क्षतिपूर्ति नीति बनाईरहेको छ। यहाँ सम्भावित यात्रा खर्चहरूको विवरण छ:
- यातायात:
तपाईँको गृह शहरदेखि गन्तव्य शहरसम्मको राउन्ड ट्रिपको हवाई भाडा।
हवाई अड्डा जान र फर्कन ट्याक्सी वा राइड-हेलिङ सेवा।
गन्तव्य शहरमा स्थानीय यातायात (जस्तै सार्वजनिक यातायात, भाडामा कार, वा ट्याक्सीहरू)।

- आवास:
बैठक स्थल नजिकको मध्य-रेंज व्यवसायिक होटेलमा तीन रातको बसाइ।

- भोजन:
कम्पनीको दैनिक भत्ता नीति अनुसार बिहानको खाजा, दिउँसोको खाना, र बेलुकाको खाना।

- विविध खर्चहरू:
हवाई अड्डामा पार्किङ्ग शुल्क।
होटेलमा इन्टरनेट पहुँच शुल्क।
उपहार वा साना सेवा शुल्कहरू।

- दस्तावेजीकरण:
तपाईँ सबै रसिदहरू (उडान, ट्याक्सी, होटेल, भोजन आदि) र भुक्तानीका लागि पूरा गरिएको खर्च रिपोर्ट पेश गर्नुहुन्छ।


## आवश्यक पुस्तकालयहरू आयात गर्नुहोस्

नोटबुकका लागि आवश्यक पुस्तकालयहरू र मोड्युलहरू आयात गर्नुहोस्।


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## खर्च मोडेलहरू परिभाषित गर्नुहोस्

 व्यक्तिगत खर्चहरूको लागि एक Pydantic मोडेल सिर्जना गर्नुहोस् र प्रयोगकर्ताको सोधपुछलाई संरचित खर्च डाटामा रूपान्तरण गर्न ExpenseFormatter वर्ग बनाउनुहोस्।

 प्रत्येक खर्च यस ढाँचामा प्रतिनिधित्व गरिनेछ:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## उपकरणहरू परिभाषित गर्दै - इमेल सिर्जना गर्दै

खर्च दाबी पेश गर्नको लागि इमेल सिर्जना गर्न एउटा उपकरण कार्य बनाउनुहोस्।
- यो उपकरण Microsoft Agent Framework बाट `@tool` डेकोरेटर प्रयोग गर्दछ।
- यसले खर्चहरूको कुल रकम गणना गर्दछ र विवरणहरूलाई इमेल शरीरमा स्वरूप प्रदान गर्दछ।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# रसिद छविहरूबाट यात्रा खर्च निकाल्ने उपकरण

यात्रा खर्च निकाल्न रसिद छविहरूबाट उपकरण कार्यक्षमता सिर्जना गर्नुहोस्।  
- यो उपकरण Microsoft Agent Framework को `@tool` डेकोरेटर प्रयोग गर्दछ।  
- यसले रसिद छवि पढ्छ, यसलाई base64 मा एनकोड गर्दछ, र एजेन्टले विश्लेषण गर्न डेटा URI फिर्ता गर्छ।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## खर्चहरूको प्रशोधन

एजेन्टहरू परिभाषित गर्नुहोस् र तिनीहरूलाई `WorkflowBuilder` प्रयोग गरी क्रमबद्ध कार्यप्रवाहमा जडान गर्नुहोस्।
- OCR एजेन्टले रिसिट छविबाट संरचित खर्च डाटा `load_receipt_image` उपकरण प्रयोग गरी निकाल्छ।
- ईमेल एजेन्टले निकालिएको डाटा लिएर `generate_expense_email` उपकरण प्रयोग गरी व्यावसायिक खर्च दाबी ईमेल तयार गर्छ।
- `WorkflowBuilder` सँग `add_edge` ले एक क्रमबद्ध पाइपलाइन बनाउँछ: OCR एजेन्ट → ईमेल एजेन्ट।


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## मुख्य कार्य

क्रमिक कार्यप्रवाह तयार गर्नुहोस् र यसलाई चलाउनुहोस् जसले रसिद छवि प्रक्रिया गरी खर्च दाबी इमेल उत्पन्न गर्दछ।


> **नोट:** यो कार्यप्रवाह हाल रिसिप्ट छविलाई बेस64 पाठको रूपमा पास गर्छ, जुन धेरै च्याट मोडेलहरूले (gpt-4o सहित) छविको रूपमा व्यवहार गर्दैनन्।  
> यसले मोडेल कन्क्टेक्स्ट विन्डो पनि अतिक्रमण गर्न सक्छ। Azure AI Vision (वा अर्को OCR उपकरण) सँग OCR चलाउने र केवल निकालिएको पाठ पास गर्ने वा छविलाई `image_url` सन्देशको रूपमा पठाउन पुनर्संरचना गर्ने प्राथमिकता दिनुहोस्।  
> यदि तपाईं केवल कन्क्टेक्स्ट त्रुटिहरूबाट बच्न चाहनुहुन्छ भने, सानो रिसिप्ट छवि वा ठूलो कन्क्टेक्स्ट विन्डो भएको मोडेल प्रयोग गर्ने प्रयास गर्नुहोस्।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
